### This notebook is entirely written with cursor/Google AI search assistance. I did not have the immediate patience to sift through the S3Tables/PyIceberg documentation. 

### This is simply a way to check and purge S3 Tables on AWS LakeForm.

In [ ]:
from FantAIno.utils.s3_utils import retrieve_s3_table_catalog
import os

In [ ]:
embeddings_catalog_name = os.getenv("S3_EMBEDDINGS_TABLE_BUCKET_NAME")
account_id = os.getenv("AWS_ACCOUNT_ID")
s3tablebucketname = os.getenv("S3_EMBEDDINGS_TABLE_BUCKET_NAME")
region = os.getenv("PYICEBERG_AWS_DEFAULT_REGION")

In [8]:
# Retrieve the S3 embeddings table catalog
embeddings_catalog = retrieve_s3_table_catalog(
    catalog_name=embeddings_catalog_name,
    account_id=account_id,
    s3tablebucketname=s3tablebucketname,
    region=region,
)

# Specify the database and table name for embeddings
embeddings_database = os.getenv("S3_EMBEDDINGS_DATABASE_NAME")
embeddings_table_name = "lyrics_embeddings_large"
embeddings_table = embeddings_catalog.load_table(f"{embeddings_database}.{embeddings_table_name}")

# Scan all records in the embeddings table and convert to Pandas DataFrame
all_records_arrow = embeddings_table.scan().to_arrow()
embeddings_df = all_records_arrow.to_pandas()

print(embeddings_df.head())

         artist_name             album_name  embedding_dim_0  embedding_dim_1  \
0           Gorillaz           The Mountain         0.004131        -0.023647   
1  Sabrina Carpenter       Mans Best Friend        -0.009102        -0.003309   
2       Flying Lotus               Big Mama        -0.000665        -0.019396   
3     Sudan Archives                THE BPM        -0.002215        -0.017995   
4               GENA  The Pleasure is Yours        -0.006113        -0.027733   

   embedding_dim_2  embedding_dim_3  embedding_dim_4  embedding_dim_5  \
0        -0.018460        -0.010770        -0.017300        -0.042578   
1        -0.022062         0.002324        -0.012032        -0.011387   
2        -0.032255         0.012753        -0.040064        -0.008174   
3        -0.021804        -0.003879        -0.024648        -0.033464   
4        -0.026387        -0.008011        -0.030287        -0.021663   

   embedding_dim_6  embedding_dim_7  ...  embedding_dim_3062  \
0        -

In [ ]:
# Warehouse must match this S3 Table bucket (not the embeddings bucket from cell 0).
album_data_s3tablebucketname = os.getenv("S3_ALBUM_DATA_TABLE_BUCKET_NAME")
album_data_catalog_name = album_data_s3tablebucketname

album_data_catalog = retrieve_s3_table_catalog(
    catalog_name=album_data_catalog_name,
    account_id=account_id,
    s3tablebucketname=album_data_s3tablebucketname,
    region=region,
)

# Database and table name for album metadata Iceberg table
album_data_database = os.getenv("S3_ALBUM_DATA_DATABASE_NAME")
album_data_table_name = "album_data"
album_data_table = album_data_catalog.load_table(f"{album_data_database}.{album_data_table_name}")

# Scan all records in the album_data table and convert to Pandas DataFrame
all_records_arrow = album_data_table.scan().to_arrow()
album_data = all_records_arrow.to_pandas()

print(album_data.head())

In [10]:
embeddings_catalog.purge_table(f"{embeddings_database}.{embeddings_table_name}")

In [ ]:
album_data_catalog.purge_table(f"{album_data_database}.{album_data_table_name}")